<a href="https://colab.research.google.com/github/HarshalMakode/Movie-Recommendation-System-using-PySpark/blob/main/MovieR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pyspark

In [19]:
!pip install streamlit pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 101.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 85.1 MB/s eta 0:00:00


In [22]:
from pyngrok import ngrok

ngrok.set_auth_token("3C20db7DcSyAJdRCQ3qL3h693Oj_4BjJo7US2cSmSbYVeSR4L")

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("MovieRecommendation") \
    .getOrCreate()

sc = spark.sparkContext

In [3]:
from google.colab import files
uploaded = files.upload()

Saving IMBD.csv to IMBD.csv


In [13]:
rdd = sc.textFile("IMBD.csv")

header = rdd.first()
rdd = rdd.filter(lambda x: x != header)

In [14]:
import csv

cleaned = rdd.map(lambda x: next(csv.reader([x]))) \
    .filter(lambda x: len(x) > 8) \
    .filter(lambda x: x[5] != '' and x[5] != 'NaN') \
    .filter(lambda x: x[8] != '' and x[8] != 'NaN')

In [15]:
top_movies = cleaned.map(lambda x: (
    x[0], float(x[5])
)).sortBy(lambda x: x[1], ascending=False)

top_movies.take(10)

[('BoJack Horseman', 9.9),
 ('1899', 9.6),
 ('Avatar: The Last Airbender', 9.6),
 ('Dexter', 9.6),
 ("JoJo's Bizarre Adventure", 9.6),
 ('Avatar: The Last Airbender', 9.6),
 ('Stranger Things', 9.6),
 ('Breaking Bad', 9.5),
 ('Avatar: The Last Airbender', 9.5),
 ('Dark', 9.5)]

In [16]:
popular = cleaned.map(lambda x: (
    x[0], int(x[8].replace(",", ""))
)).sortBy(lambda x: x[1], ascending=False)

popular.take(10)

[('The Lord of the Rings: The Fellowship of the Ring', 1844075),
 ('The Lord of the Rings: The Fellowship of the Ring', 1844055),
 ('Breaking Bad', 1831359),
 ('Breaking Bad', 1831340),
 ('The Lord of the Rings: The Return of the King', 1819157),
 ('The Lord of the Rings: The Two Towers', 1642708),
 ('Gladiator', 1481531),
 ('The Departed', 1310171),
 ('Titanic', 1158746),
 ('Stranger Things', 1149902)]

In [17]:
genre_ratings = cleaned.flatMap(lambda x: [
    (g.strip(), (float(x[5]), 1)) for g in x[4].split(",")
])

avg_genre = genre_ratings.reduceByKey(
    lambda a, b: (a[0]+b[0], a[1]+b[1])
).mapValues(lambda x: x[0]/x[1])

avg_genre.collect()

[('Action', 6.8845070422535235),
 ('Comedy', 6.701560837176296),
 ('Drama', 6.9067424643046005),
 ('History', 7.267065868263472),
 ('Horror', 5.803232323232324),
 ('Adventure', 7.073453237410071),
 ('Romance', 6.699352750809062),
 ('Sport', 6.849382716049382),
 ('Sci-Fi', 6.341015624999999),
 ('Documentary', 7.02047697368421),
 ('Music', 6.859813084112147),
 ('Musical', 6.749333333333334),
 ('', 7.175),
 ('Biography', 6.965588235294118),
 ('Crime', 6.88153409090909),
 ('Mystery', 6.8547814207650255),
 ('Animation', 7.1715624999999985),
 ('Fantasy', 6.927016129032259),
 ('Thriller', 6.129746835443039),
 ('Short', 6.6251999999999995),
 ('War', 6.9322580645161285),
 ('Family', 6.6544364508393254),
 ('Reality-TV', 6.891334894613588),
 ('Western', 6.5290322580645155),
 ('Game-Show', 6.968),
 ('Talk-Show', 7.06842105263158),
 ('Film-Noir', 6.966666666666666),
 ('News', 7.068421052631578)]

In [28]:
%%writefile app.py
import streamlit as st
import csv
import matplotlib.pyplot as plt

st.set_page_config(page_title="Movie Recommender", layout="wide")

st.title("🎬 Movie Recommendation System")
st.write("Built with PySpark Logic + Streamlit UI 🚀")

# Upload file
uploaded_file = st.file_uploader("📂 Upload IMBD.csv", type=["csv"])

if uploaded_file:
    data = uploaded_file.read().decode("utf-8").splitlines()
    reader = list(csv.reader(data))

    header = reader[0]
    rows = reader[1:]

    # Clean data
    cleaned = []
    for row in rows:
        try:
            if len(row) > 8 and row[5] != '' and row[8] != '':
                cleaned.append(row)
        except:
            continue

    st.success(f"✅ Loaded {len(cleaned)} records")

    # Sidebar filters
    st.sidebar.header("🔍 Filters")

    # Genre filter
    all_genres = set()
    for row in cleaned:
        for g in row[4].split(","):
            all_genres.add(g.strip())

    genre_choice = st.sidebar.selectbox("Select Genre", ["All"] + sorted(all_genres))

    # Search box
    search = st.sidebar.text_input("Search Movie")

    # Filter data
    filtered = cleaned

    if genre_choice != "All":
        filtered = [row for row in filtered if genre_choice.lower() in row[4].lower()]

    if search:
        filtered = [row for row in filtered if search.lower() in row[0].lower()]

    # Tabs
    tab1, tab2, tab3 = st.tabs(["⭐ Top Rated", "🔥 Popular", "📊 Genre Analysis"])

    # ⭐ Top Rated
    with tab1:
        st.subheader("Top Rated Movies")

        movies = []
        for row in filtered:
            try:
                movies.append((row[0], float(row[5])))
            except:
                continue

        movies = sorted(movies, key=lambda x: x[1], reverse=True)

        for m in movies[:10]:
            st.write(f"🎬 {m[0]} ⭐ {m[1]}")

    # 🔥 Popular Movies
    with tab2:
        st.subheader("Most Popular Movies")

        movies = []
        for row in filtered:
            try:
                votes = int(row[8].replace(",", ""))
                movies.append((row[0], votes))
            except:
                continue

        movies = sorted(movies, key=lambda x: x[1], reverse=True)

        for m in movies[:10]:
            st.write(f"🎬 {m[0]} 👥 {m[1]} votes")

    # 📊 Genre Analysis
    with tab3:
        st.subheader("Average Rating by Genre")

        genre_dict = {}

        for row in cleaned:
            try:
                rating = float(row[5])
                genres = row[4].split(",")

                for g in genres:
                    g = g.strip()
                    if g not in genre_dict:
                        genre_dict[g] = [0, 0]
                    genre_dict[g][0] += rating
                    genre_dict[g][1] += 1
            except:
                continue

        genres = []
        ratings = []

        for g in genre_dict:
            avg = genre_dict[g][0] / genre_dict[g][1]
            genres.append(g)
            ratings.append(avg)

        fig, ax = plt.subplots()
        ax.bar(genres, ratings)
        plt.xticks(rotation=45)

        st.pyplot(fig)

else:
    st.info("👆 Upload your IMBD.csv file to start")

Overwriting app.py


In [29]:
!streamlit run app.py &>/dev/null &

In [30]:
public_url = ngrok.connect(8501)
public_url

<NgrokTunnel: "https://immature-lily-casually.ngrok-free.dev" -> "http://localhost:8501">